In [13]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from PIL import Image
import os

def detect_background_color(img, samples_per_edge=100):
    """
    Safely detect the background color by analyzing image edges.
    Uses a more robust sampling approach that avoids zero-step slicing.
    
    Args:
        img: Input image in BGR format
        samples_per_edge: Number of samples to take from each edge
        
    Returns:
        tuple: RGB color values of detected background
    """
    height, width = img.shape[:2]
    
    # Ensure we take at least one sample per edge
    samples_per_edge = max(1, min(samples_per_edge, min(height, width)))
    
    # Calculate step sizes (ensure non-zero)
    width_step = max(1, width // samples_per_edge)
    height_step = max(1, height // samples_per_edge)
    
    # Sample pixels from edges
    edge_pixels = []
    
    # Sample top and bottom edges
    for x in range(0, width, width_step):
        edge_pixels.append(img[0, x])        # Top edge
        edge_pixels.append(img[-1, x])       # Bottom edge
    
    # Sample left and right edges
    for y in range(0, height, height_step):
        edge_pixels.append(img[y, 0])        # Left edge
        edge_pixels.append(img[y, -1])       # Right edge
    
    # Convert to numpy array
    edge_pixels = np.array(edge_pixels)
    
    # Use K-means clustering to find dominant color
    kmeans = KMeans(n_clusters=3, n_init=10)
    kmeans.fit(edge_pixels)
    
    # Get the most frequent color cluster
    unique, counts = np.unique(kmeans.labels_, return_counts=True)
    dominant_cluster = unique[np.argmax(counts)]
    background_color = kmeans.cluster_centers_[dominant_cluster].astype(int)
    
    # Convert from BGR to RGB
    return tuple(background_color[::-1])

def detect_shadows_enhanced(img, background_color):
    """
    Enhanced shadow detection that preserves subtle lighting transitions and reflections.
    This function analyzes both global and local lighting patterns to identify shadows
    while being careful not to misclassify object details as shadows.
    
    Args:
        img: Input image in BGR format
        background_color: RGB tuple of the background color
    Returns:
        numpy.ndarray: Shadow mask where 1 indicates shadow pixels
    """
    # First, convert background color from RGB to BGR for OpenCV
    bg_color = background_color[::-1]
    
    # Create a background image matching our detected background color
    background = np.full_like(img, bg_color)
    
    # Convert both images to LAB color space for better lighting analysis
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    bg_lab = cv2.cvtColor(background, cv2.COLOR_BGR2LAB)
    
    # Extract L (lightness) channels
    l_img = lab[:,:,0].astype(np.float32)
    l_bg = bg_lab[:,:,0].astype(np.float32)
    
    # Calculate local average of lightness using multiple scales
    shadow_masks = []
    for kernel_size in [(21, 21), (41, 41)]:  # Multiple scales for different shadow sizes
        # Calculate local lightness average
        local_mean = cv2.GaussianBlur(l_img, kernel_size, 0)
        
        # Calculate lightness difference from background
        l_diff = np.abs(l_img - l_bg)
        local_diff = np.abs(local_mean - l_bg)
        
        # Create adaptive threshold based on local contrast
        threshold = np.mean(l_diff) * 0.5 + local_diff * 0.2
        
        # Identify shadow regions
        shadow = (l_diff < threshold) & (l_img < l_bg)
        shadow_masks.append(shadow)
    
    # Combine shadow masks from different scales
    shadow_mask = np.logical_or.reduce(shadow_masks)
    
    # Analyze color differences to prevent misclassifying colored regions as shadows
    a_diff = np.abs(lab[:,:,1] - bg_lab[:,:,1])
    b_diff = np.abs(lab[:,:,2] - bg_lab[:,:,2])
    color_diff = np.sqrt(a_diff**2 + b_diff**2)
    
    # Only keep shadow pixels where color difference is small
    shadow_mask &= (color_diff < 30)
    
    # Clean up the mask
    kernel = np.ones((3,3), np.uint8)
    shadow_mask = cv2.morphologyEx(shadow_mask.astype(np.uint8), 
                                 cv2.MORPH_CLOSE, kernel)
    shadow_mask = cv2.morphologyEx(shadow_mask.astype(np.uint8), 
                                 cv2.MORPH_OPEN, kernel)
    
    return shadow_mask

def refined_background_removal(image_input, background_color=None, edge_smoothing=5, preserve_whites=True):
    """
    Advanced background removal with automatic background color detection and
    improved edge handling.
    
    Args:
        image_input: Path to input image or Image.Image or numpy array
        background_color: RGB tuple or None for auto-detection
        edge_smoothing: Amount of edge smoothing (higher = smoother)
        preserve_whites: Whether to preserve white colors in the object
    """
    def create_color_range_mask(img, target_color, tolerance=20):  # Reduced tolerance
        """Create a more precise mask for colors within tolerance of target color"""
        target_bgr = target_color[::-1]

        # Convert to LAB color space for better color similarity matching
        lab_image = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        lab_target = cv2.cvtColor(np.uint8([[target_bgr]]), cv2.COLOR_BGR2LAB)[0,0]

        # Create bounds with tolerance in LAB space
        lower_bound = np.array([max(0, c - tolerance) for c in lab_target])
        upper_bound = np.array([min(255, c + tolerance) for c in lab_target])

        # Create mask in LAB space
        mask = cv2.inRange(lab_image, lower_bound, upper_bound)

        # Clean up the mask
        kernel = np.ones((3,3), np.uint8)
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        return mask

    def smooth_edges(mask, smooth_factor):
        """Apply edge smoothing with outline removal"""
        # Convert to float
        mask_float = mask.astype(np.float32) / 255.0
        
        # Apply erosion first to remove thin outlines
        kernel = np.ones((2,2), np.uint8)
        eroded = cv2.erode(mask_float, kernel, iterations=1)
        
        # Multi-scale smoothing
        smoothed = np.zeros_like(mask_float)
        weights_sum = 0
        
        for i in range(1, 4):
            kernel_size = smooth_factor * 2 * i + 1
            current_smooth = cv2.GaussianBlur(eroded, 
                                            (kernel_size, kernel_size), 
                                            0)
            weight = 1.0 / i
            smoothed += current_smooth * weight
            weights_sum += weight
        
        smoothed /= weights_sum
        
        # Threshold the result to make edges cleaner
        smoothed = np.where(smoothed > 0.5, 1.0, 0.0)
        
        return (smoothed * 255).astype(np.uint8)

    def preserve_white_details(img, mask, threshold=250):
        """
        Preserve white details with proper handling of edge cases and division.
        
        Args:
            img: Input image in BGR format
            mask: Binary mask
            threshold: Brightness threshold for white detection
        """
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Create adaptive threshold with safety checks
        local_mean = cv2.GaussianBlur(gray, (15, 15), 0)
        
        # Avoid division by zero and invalid values
        local_mean = np.clip(local_mean, 1, 254)  # Ensure no zeros or 255s
        adjustment = np.clip((255 - local_mean) * 0.05, 0, threshold)
        local_threshold = threshold - adjustment
        
        # Create white mask with safety checks
        white_areas = gray > local_threshold
        
        # Reduce connectivity to main object
        kernel = np.ones((2,2), np.uint8)
        dilated_mask = cv2.dilate(mask, kernel, iterations=1)
        preserved_whites = white_areas & dilated_mask
        
        # Clean up artifacts
        preserved_whites = cv2.morphologyEx(preserved_whites.astype(np.uint8), 
                                        cv2.MORPH_OPEN, kernel)
        preserved_whites = cv2.morphologyEx(preserved_whites, 
                                        cv2.MORPH_CLOSE, kernel)
        
        return mask | preserved_whites

    def shrink_mask(mask, shrink_percent=1):
        """
        Shrink the mask by a percentage of its dimensions
        Args:
            mask: Binary mask
            shrink_percent: Percentage to shrink (1 = 1%)
        Returns:
            Shrunk mask
        """
        # Get mask dimensions
        height, width = mask.shape[:2]
        
        # Calculate pixels to shrink on each side
        shrink_pixels_y = int(height * (shrink_percent / 100))
        shrink_pixels_x = int(width * (shrink_percent / 100))
        
        # Ensure at least 1 pixel if percentage is too small
        shrink_pixels_y = max(1, shrink_pixels_y)
        shrink_pixels_x = max(1, shrink_pixels_x)
        
        # Create structuring element for erosion
        kernel = cv2.getStructuringElement(
            cv2.MORPH_ELLIPSE,
            (2 * shrink_pixels_x + 1, 2 * shrink_pixels_y + 1)
        )
        
        # Erode the mask
        shrunk_mask = cv2.erode(mask, kernel, iterations=1)
        
        return shrunk_mask

    # Main processing pipeline
    try:
        print("Loading image...")
        # Input validation and conversion
        if isinstance(image_input, str):
            print("It's a file path - load the image")
            if not os.path.exists(image_input):
                raise ValueError(f"Image file not found: {image_input}")
            image = cv2.imread(image_input)
            if image is None:
                raise ValueError(f"Failed to load image from {image_input}")
        elif isinstance(image_input, np.ndarray):
            print("It's already a numpy array - verify format")
            if len(image_input.shape) != 3 or image_input.shape[2] != 3:
                raise ValueError("Image array must be a 3-channel color image")
            image = image_input
        elif isinstance(image_input, Image.Image):
            
            print("Convert PIL Image to numpy array in BGR format")
            image_array = np.array(image_input)
            image = cv2.cvtColor(image_array, cv2.COLOR_RGB2BGR)
        else:
            raise TypeError("Image input must be either a file path, numpy array, or PIL Image")
        
        print(image.__class__)
        # Handle background color detection
        if background_color is None:
            print("Detecting background color...")
            detected_color = detect_background_color(image)
            print(f"Detected background color (RGB): {detected_color}")
            bg_color = detected_color
        else:
            bg_color = background_color
        
        print("Detecting shadows...")
        shadow_mask = detect_shadows_enhanced(image, bg_color)
        
        print("Creating color-based mask...")
        color_mask = create_color_range_mask(image, bg_color)
        
        # Combine color and shadow masks
        combined_mask = (color_mask | shadow_mask)

        
        
        # Invert mask (we want to keep the object, not the background)
        object_mask = cv2.bitwise_not(combined_mask)
        
        # Preserve white details if requested
        if preserve_whites:
            print("Preserving white details...")
            object_mask = preserve_white_details(image, object_mask)

        
        # Apply edge smoothing
        if edge_smoothing > 0:
            print("Smoothing edges...")
            object_mask = smooth_edges(object_mask, edge_smoothing)

        # Shrinking mask
        print("Shrinking mask...")
        object_mask = shrink_mask(object_mask, shrink_percent=0.5)
        
        
        # Create output images
        alpha = object_mask
        rgba = cv2.cvtColor(image, cv2.COLOR_BGR2BGRA)
        rgba[:, :, 3] = alpha
        
        result = image.copy()
        result[alpha == 0] = [0, 0, 0]
        
        # Convert to RGB for display
        image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
        
        checkered = np.zeros((image.shape[0], image.shape[1], 3), dtype=np.uint8)
        checkered[::20, ::20] = [200, 200, 200]
        checkered[10::20, 10::20] = [200, 200, 200]
        
        alpha_3d = alpha[:,:,np.newaxis] / 255.0
        blended = (image_rgb * alpha_3d + checkered * (1 - alpha_3d)).astype(np.uint8)
        
        return image_rgb, alpha, result_rgb, blended, rgba
        
    except Exception as e:
        print(f"Error during processing: {str(e)}")
        return None

In [20]:
front_results = refined_background_removal(
            "../static/uploads/Chair/front.jpg",
            background_color=None,  # Enable auto-detection
            edge_smoothing=5,
            preserve_whites=True
        )

front_feature = front_results[2];

back_results = refined_background_removal(
            "../static/uploads/Chair/back.jpg",
            background_color=None,  # Enable auto-detection
            edge_smoothing=5,
            preserve_whites=True
        )

back_feature = back_results[2];
top_results = refined_background_removal(
            "../static/uploads/Chair/top.jpg",
            background_color=None,  # Enable auto-detection
            edge_smoothing=5,
            preserve_whites=True
        )

top_feature = top_results[2];
top_binary_mask = top_results[1];

Loading image...
It's a file path - load the image
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...
Loading image...
It's a file path - load the image
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.

c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.



Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...
Loading image...
It's a file path - load the image
<class 'numpy.ndarray'>
Detecting background color...
Detected background color (RGB): (255, 255, 255)
Detecting shadows...
Creating color-based mask...
Preserving white details...
Smoothing edges...
Shrinking mask...


c:\Users\ACER\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\base.py:1474: ConvergenceWarning:

Number of distinct clusters (1) found smaller than n_clusters (3). Possibly due to duplicate points in X.



In [21]:
import torch
import os
import cv2
import numpy as np
from PIL import Image
import plotly.graph_objects as go
from transformers import pipeline
 
# Get depth map using Depth Anything model
device = "cuda" if torch.cuda.is_available() else "cpu"
depth_model = pipeline("depth-estimation", model="depth-anything/Depth-Anything-V2-base-hf", device=device)

In [22]:
def get_depth_map(result_rgb):
    # Use result_rgb directly for depth estimation (it's already in RGB format)
    image_pil = Image.fromarray(result_rgb)
    depth_predictions = depth_model(image_pil)
    depth_map = np.array(depth_predictions["depth"])

    # Process depth map with dimensions from result_rgb
    height, width = result_rgb.shape[:2]
    depth_map_resized = cv2.resize(depth_map, (width, height))

    return depth_map_resized

front_depth_map = get_depth_map(front_feature)
back_depth_map = get_depth_map(back_feature)
top_depth_map = get_depth_map(top_feature)

In [23]:
def create_dynamic_scatter_visualization(depth_map, image, title, target_size=100):
    """
    Creates a 3D scatter visualization where the number of vertical points
    is determined dynamically based on the depth range at each location.
    """
    # Normalize dimensions as before
    height, width = depth_map.shape
    scale_factor = target_size / max(width, height)
    
    new_width = int(width * scale_factor)
    new_height = int(height * scale_factor)
    
    # Resize our maps
    depth_norm = cv2.resize(depth_map, (new_width, new_height))
    image_resized = cv2.resize(image, (new_width, new_height))
    
    # Create mask for valid points
    valid_mask = ~np.all(image_resized == 0, axis=2)
    
    # Normalize depth values
    depth_norm = depth_norm / depth_norm.max() * target_size
    
    # Find the global minimum depth (highest point in 3D space) among valid points
    global_min_depth = np.min(depth_norm[valid_mask])
    
    # Initialize our point collections
    x_all = []
    y_all = []
    z_all = []
    colors_all = []
    
    # Process each valid point
    for i in range(new_height):
        for j in range(new_width):
            if valid_mask[i, j]:
                # Get the current point's depth
                current_depth = depth_norm[i, j]
                
                # Calculate how many points we need for this vertical line
                # The number of points is proportional to the distance from the global minimum
                depth_difference = current_depth - global_min_depth
                
                # We'll use one point per unit of depth difference
                # This ensures we have exactly the right number of points needed
                num_points = max(1, int(depth_difference))
                
                if num_points > 1:  # Only create vertical line if there's a meaningful depth difference
                    # Create evenly spaced points from current depth to the global minimum
                    z_points = np.linspace(current_depth, global_min_depth, num_points)
                    
                    # Add these points to our collections
                    x_all.extend([j] * num_points)
                    y_all.extend([i] * num_points)
                    z_all.extend(z_points)
                    
                    # Use the original color for all points in this line
                    original_color = image_resized[i, j]
                    colors_all.extend([original_color] * num_points)
                else:
                    # For points at or near the minimum depth, just add a single point
                    x_all.append(j)
                    y_all.append(i)
                    z_all.append(current_depth)
                    colors_all.append(image_resized[i, j])
    
    # Convert to numpy arrays
    x_all = np.array(x_all)
    y_all = np.array(y_all)
    z_all = np.array(z_all)
    colors_all = np.array(colors_all)
    
    # Create color strings
    color_strings = [f'rgb({r},{g},{b})' for r,g,b in colors_all]
    
    # Create the visualization
    fig = go.Figure(data=[go.Scatter3d(
        x=x_all,
        y=y_all,
        z=z_all,
        mode='markers',
        marker=dict(
            size=2,
            color=color_strings,
            opacity=1.0
        )
    )])
    
    # Update layout
    fig.update_layout(
        title=dict(
            text=f"{title}\nPoints distributed based on depth difference",
            x=0.5,
            y=0.95,
            xanchor='center',
            yanchor='top',
            font=dict(size=20)
        ),
        scene=dict(
            xaxis=dict(
                range=[0, new_width],
                nticks=10,
                title='Width',
                gridcolor='lightgray',
                showbackground=True,
                backgroundcolor='white'
            ),
            yaxis=dict(
                range=[0, new_height],
                nticks=10,
                title='Height',
                gridcolor='lightgray',
                showbackground=True,
                backgroundcolor='white'
            ),
            zaxis=dict(
                range=[0, target_size],
                nticks=10,
                title='Depth',
                gridcolor='lightgray',
                showbackground=True,
                backgroundcolor='white'
            )
        ),
        width=800,
        height=800,
        showlegend=False,
        paper_bgcolor='white'
    )
    
    return fig, len(x_all)  # Return point count for information

In [30]:
def combine_meshes_with_top_profile(front_fig, back_fig, top_fig):
    """
    Combines front and back meshes using the top view as the definitive reference 
    for spatial distribution. This ensures the combined view matches the top-down perspective.
    """
    def extract_mesh_points(fig):
        """
        Extracts points and their colors from a Plotly mesh figure while preserving 
        the crucial spatial relationships.
        """
        trace = fig.data[0]
        points = np.column_stack((
            np.array(trace.x),
            np.array(trace.y),
            np.array(trace.z)
        ))
        colors = np.array([c.strip('rgb()').split(',') for c in trace.marker.color])
        return points, colors
    
    def analyze_top_profile(top_points, resolution=100):
        """
        Creates a detailed mapping of width distribution from the top view.
        At each height level, we calculate both the width and the center line.
        """
        height_levels = np.linspace(np.min(top_points[:, 1]), 
                                  np.max(top_points[:, 1]), 
                                  resolution)
        profile = []
        
        for height in height_levels:
            # Get points at this height level
            slice_mask = np.abs(top_points[:, 1] - height) < (height_levels[1] - height_levels[0])
            slice_points = top_points[slice_mask]
            
            if len(slice_points) > 0:
                # In top view, x coordinates represent width distribution
                left_edge = np.min(slice_points[:, 0])
                right_edge = np.max(slice_points[:, 0])
                center_line = (left_edge + right_edge) / 2
                width = right_edge - left_edge
                
                profile.append({
                    'height': height,
                    'center': center_line,
                    'width': width,
                    'left': left_edge,
                    'right': right_edge
                })
        
        return profile
    
    def redistribute_points(points, profile, is_front=True):
        """
        Redistributes points to match the top view profile while maintaining
        relative spatial relationships within each view.
        """
        from scipy.interpolate import interp1d
        
        # Create interpolation functions for the profile
        heights = np.array([p['height'] for p in profile])
        centers = np.array([p['center'] for p in profile])
        widths = np.array([p['width'] for p in profile])
        
        center_interpolator = interp1d(heights, centers, kind='linear', 
                                     fill_value='extrapolate')
        width_interpolator = interp1d(heights, widths, kind='linear',
                                    fill_value='extrapolate')
        
        # Adjust each point's position
        adjusted_points = points.copy()
        for i in range(len(points)):
            height = points[i, 1]
            
            # Get profile information for this height
            local_width = width_interpolator(height)
            local_center = center_interpolator(height)
            
            # Calculate relative position in original depth
            depth_range = np.max(points[:, 2]) - np.min(points[:, 2])
            relative_depth = (points[i, 2] - np.min(points[:, 2])) / depth_range
            
            # Adjust z-coordinate (depth) based on top view profile
            if is_front:
                adjusted_points[i, 2] = local_center + (local_width/2) * relative_depth
            else:
                adjusted_points[i, 2] = local_center - (local_width/2) * relative_depth
        
        return adjusted_points
    
    # Extract points from all views
    front_points, front_colors = extract_mesh_points(front_fig)
    back_points, back_colors = extract_mesh_points(back_fig)
    top_points, _ = extract_mesh_points(top_fig)
    
    # Analyze the top view profile
    profile = analyze_top_profile(top_points)
    
    # Redistribute front and back points according to top view
    front_adjusted = redistribute_points(front_points, profile, is_front=True)
    back_adjusted = redistribute_points(back_points, profile, is_front=False)
    
    # Combine the adjusted points
    combined_points = np.vstack([front_adjusted, back_adjusted])
    combined_colors = [f'rgb({r},{g},{b})' for r,g,b in np.vstack([front_colors, back_colors])]
    
    # Create the combined visualization
    combined_fig = go.Figure(data=[go.Scatter3d(
        x=combined_points[:, 0],
        y=combined_points[:, 1],
        z=combined_points[:, 2],
        mode='markers',
        marker=dict(
            size=2,
            color=combined_colors,
            opacity=1
        )
    )])
    
    # Update layout with better viewing angles
    combined_fig.update_layout(
        scene=dict(
            aspectmode='data',
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.5),
                up=dict(x=0, y=0, z=1)
            )
        ),
        showlegend=False
    )
    
    return combined_fig

In [ ]:
# First, let's get our three normalized meshes individually
front_fig, front_length = create_dynamic_scatter_visualization(front_depth_map, front_feature, "Front View")
back_fig, back_length = create_dynamic_scatter_visualization(back_depth_map, back_feature, "Back View")
top_fig, top_length = create_dynamic_scatter_visualization(top_depth_map, top_feature, "Top View")

# front_fig.show()
# back_fig.show()
# top_fig.show()

# For maximum quality (but slower)
combined_mesh = combine_meshes_with_top_profile(front_fig, back_fig, top_fig= top_fig)
combined_mesh.show()